# String Distance, Similarity and Evaluation Metrics

This notebook accompanies the lecture on measuring text similarity at different levels: character-based string distance, semantic similarity using embeddings, and NLP system evaluation metrics.

## Learning Objectives
By the end of this notebook, you will be able to:
1. Compute and apply various string distance metrics (Levenshtein, Hamming, Jaro-Winkler)
2. Use modern semantic similarity approaches with sentence embeddings (SBERT)
3. Evaluate NLP systems using metrics like BLEU, ROUGE, and others
4. Choose the appropriate metric for different NLP tasks

## Setup and Installation

First, let's install the required libraries:

In [ ]:
# Install required packages
!pip install python-Levenshtein
!pip install jellyfish
!pip install sentence-transformers
!pip install rouge-score
!pip install nltk
!pip install sacrebleu
!pip install matplotlib
!pip install seaborn
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install transformers
!pip install bert-score

In [ ]:
# Import all necessary libraries
import Levenshtein
import jellyfish
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import sacrebleu
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK data
nltk.download('punkt', quiet=True)

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Part 1: String Distance Metrics

String distance metrics measure **dissimilarity** between strings by counting the minimum number of operations needed to transform one string into another. These operate at the **character level** and focus on **lexical similarity**, not semantic meaning.

## Levenshtein Distance (Edit Distance)

The **Levenshtein distance** (also known as edit distance) is the minimum number of single-character edits (insertions, deletions, or substitutions) required to change one string into another.

### Algorithm Overview
The Levenshtein distance uses dynamic programming with time complexity **O(m×n)** where m and n are the lengths of the two strings.

### Use Cases
- Spell checking: "recieve" → "receive"
- OCR correction
- Fuzzy string matching
- DNA sequence alignment

In [ ]:
def visualize_levenshtein(str1, str2):
    """
    Calculate and visualize the Levenshtein distance between two strings.
    """
    distance = Levenshtein.distance(str1, str2)
    ratio = Levenshtein.ratio(str1, str2)

    print(f"String 1: '{str1}'")
    print(f"String 2: '{str2}'")
    print(f"Levenshtein Distance: {distance}")
    print(f"Similarity Ratio: {ratio:.3f}")
    print(f"")

    # Show the edit operations
    opcodes = Levenshtein.opcodes(str1, str2)
    print("Edit Operations:")
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'replace':
            print(f"  Replace '{str1[i1:i2]}' with '{str2[j1:j2]}'")
        elif tag == 'delete':
            print(f"  Delete '{str1[i1:i2]}'")
        elif tag == 'insert':
            print(f"  Insert '{str2[j1:j2]}'")
        elif tag == 'equal':
            print(f"  Keep '{str1[i1:i2]}'")

    return distance, ratio

In [ ]:
# Example 1: Simple typo correction
print("=" * 60)
print("Example 1: Typo Correction")
print("=" * 60)
visualize_levenshtein("recieve", "receive")

print("\n" + "=" * 60)
print("Example 2: OCR Error")
print("=" * 60)
visualize_levenshtein("telebision", "television")

print("\n" + "=" * 60)
print("Example 3: Phone Brand Typo")
print("=" * 60)
visualize_levenshtein("ifone", "iphone")

### Levenshtein Distance Matrix Visualization

Let's visualize the dynamic programming matrix used to compute the Levenshtein distance:

In [ ]:
def levenshtein_matrix(str1, str2):
    """
    Compute the Levenshtein distance matrix (dynamic programming table).
    """
    m, n = len(str1), len(str2)

    # Create matrix
    matrix = np.zeros((m + 1, n + 1), dtype=int)

    # Initialize first row and column
    for i in range(m + 1):
        matrix[i][0] = i
    for j in range(n + 1):
        matrix[0][j] = j

    # Fill the matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i-1] == str2[j-1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i-1][j] + 1,      # deletion
                matrix[i][j-1] + 1,      # insertion
                matrix[i-1][j-1] + cost  # substitution
            )

    return matrix

def plot_levenshtein_matrix(str1, str2):
    """
    Visualize the Levenshtein distance matrix as a heatmap.
    """
    matrix = levenshtein_matrix(str1, str2)

    fig, ax = plt.subplots(figsize=(max(8, len(str2)), max(6, len(str1))))

    # Create heatmap
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Reds',
                xticklabels=[' '] + list(str2),
                yticklabels=[' '] + list(str1),
                cbar_kws={'label': 'Edit Distance'},
                linewidths=0.5, linecolor='gray', ax=ax)

    ax.set_xlabel('String 2', fontsize=12, fontweight='bold')
    ax.set_ylabel('String 1', fontsize=12, fontweight='bold')
    ax.set_title(f'Levenshtein Distance Matrix: "{str1}" → "{str2}"\nFinal Distance: {matrix[-1, -1]}',
                 fontsize=14, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.show()

    return matrix

# Visualize the matrix for our example
plot_levenshtein_matrix("kitten", "sitting");

## Practical Application: Spell Checking

Let's build a simple spell checker using string distance metrics.

In [ ]:
class SimpleSpellChecker:
    def __init__(self, vocabulary, max_distance=2):
        """
        Initialize spell checker with a vocabulary.

        Args:
            vocabulary: List of correct words
            max_distance: Maximum edit distance to consider
        """
        self.vocabulary = set(vocabulary)
        self.max_distance = max_distance

    def suggest_corrections(self, word, n=5):
        """
        Suggest corrections for a potentially misspelled word.
        """
        if word in self.vocabulary:
            return [(word, 0, 1.0)]  # Already correct

        # Calculate distances to all vocabulary words
        candidates = []
        for vocab_word in self.vocabulary:
            distance = Levenshtein.distance(word, vocab_word)
            if distance <= self.max_distance:
                ratio = Levenshtein.ratio(word, vocab_word)
                candidates.append((vocab_word, distance, ratio))

        # Sort by distance (ascending) and ratio (descending)
        candidates.sort(key=lambda x: (x[1], -x[2]))

        return candidates[:n]

    def check_and_suggest(self, word):
        """
        Check a word and print suggestions if misspelled.
        """
        print(f"Checking: '{word}'")
        suggestions = self.suggest_corrections(word)

        if not suggestions:
            print("  No suggestions found (word too different from vocabulary)\n")
            return

        if suggestions[0][1] == 0:
            print("  Correct spelling\n")
            return

        print("  Possible misspelling. Suggestions:")
        for i, (suggestion, distance, ratio) in enumerate(suggestions, 1):
            print(f"    {i}. {suggestion:15s} (distance: {distance}, similarity: {ratio:.3f})")
        print()

# Create a simple vocabulary
vocabulary = [
    "receive", "believe", "achieve", "deceive",
    "television", "telephone", "telegram", "telescope",
    "iphone", "phone", "smartphone", "cellphone",
    "computer", "laptop", "desktop", "tablet",
    "python", "java", "javascript", "programming"
]

spell_checker = SimpleSpellChecker(vocabulary, max_distance=3)

# Test the spell checker
test_words = ["recieve", "telebision", "ifone", "programing", "receive", "smartfone"]

for word in test_words:
    spell_checker.check_and_suggest(word)

## Limitations of String Distance Metrics

**Important**: String distance metrics operate at the character or word level and **cannot capture semantic meaning**.

Let's see this limitation in action:

In [ ]:
# Demonstrating the semantic limitation

# Semantically similar but lexically different
sent1 = "The cat sat on the mat"
sent2 = "A kitty rested on the rug"
sent3 = "The cat sat on the hat"  # Lexically similar but semantically different

print(f"Sentence 1: {sent1}")
print(f"Sentence 2: {sent2}")
print(f"Sentence 3: {sent3}")
print()

# Compare distances
dist_1_2 = Levenshtein.distance(sent1, sent2)
ratio_1_2 = Levenshtein.ratio(sent1, sent2)
dist_1_3 = Levenshtein.distance(sent1, sent3)
ratio_1_3 = Levenshtein.ratio(sent1, sent3)

print(f"Sent1 vs Sent2 (semantically similar, lexically different):")
print(f"  Levenshtein Distance: {dist_1_2}")
print(f"  Similarity Ratio: {ratio_1_2:.3f}")
print()

print(f"Sent1 vs Sent3 (semantically different, lexically similar):")
print(f"  Levenshtein Distance: {dist_1_3}")
print(f"  Similarity Ratio: {ratio_1_3:.3f}")
print()

String distance considers Sent1 and Sent3 MORE similar even though Sent1 and Sent2 have the same meaning!

The solution is to use semantic similarity metrics.

# Part 2: Semantic Similarity with Sentence Embeddings

Unlike string distance metrics, **semantic similarity** measures how similar two pieces of text are in **meaning**, regardless of exact wording.

Modern approaches use **sentence embeddings** - dense vector representations that capture semantic meaning. We'll use **Sentence-BERT (SBERT)**, a modification of BERT specifically trained for sentence similarity tasks.

## Introduction to Sentence-BERT (SBERT)

**Key Concepts:**
- SBERT creates fixed-size embeddings for sentences
- Trained using siamese/triplet networks to produce semantically meaningful embeddings
- Similarity is measured using **cosine similarity** between embeddings
- Much faster than BERT for finding similar sentences

**Cosine Similarity:**
$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{||A|| \cdot ||B||}$$

Values range from -1 (opposite) to 1 (identical).

In [ ]:
# Load a pre-trained SBERT model
# We'll use 'all-MiniLM-L6-v2'. A good balance of speed and performance
print("Loading Sentence-BERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!\n")

# Get embedding size
test_embedding = sbert_model.encode("test")
print(f"Embedding dimension: {len(test_embedding)}")

## Computing Semantic Similarity

In [ ]:
def compute_semantic_similarity(sent1, sent2, model=sbert_model):
    """
    Compute semantic similarity between two sentences using SBERT.
    """
    # Encode sentences
    emb1 = model.encode(sent1, convert_to_tensor=True)
    emb2 = model.encode(sent2, convert_to_tensor=True)

    # Compute cosine similarity
    similarity = util.cos_sim(emb1, emb2).item()

    return similarity, emb1, emb2

def compare_similarities(sent1, sent2):
    """
    Compare both string distance and semantic similarity.
    """
    print(f"Sentence 1: {sent1}")
    print(f"Sentence 2: {sent2}")
    print("=" * 70)

    # String distance
    lev_ratio = Levenshtein.ratio(sent1, sent2)
    print(f"Levenshtein Ratio:     {lev_ratio:.3f}")

    # Semantic similarity
    sem_sim, _, _ = compute_semantic_similarity(sent1, sent2)
    print(f"Semantic Similarity:   {sem_sim:.3f}")
    print()

# Example: The cat/feline example from earlier
print("Comparing string distance vs semantic similarity\n")
compare_similarities(
    "The cat sat on the mat",
    "A kitty rested on the rug"
)

compare_similarities(
    "The cat sat on the mat",
    "The cat sat on the hat"
)

As it can be seen, semantic similarity correctly identifies meaning.

## Practical Example: Semantic Search

One powerful application of semantic similarity is **semantic search**: finding relevant documents based on meaning rather than exact keyword matches.

In [ ]:
class SemanticSearch:
    def __init__(self, documents, model=sbert_model):
        """
        Initialize semantic search with a corpus of documents.

        Args:
            documents: List of text documents
            model: Sentence transformer model
        """
        self.documents = documents
        self.model = model

        # Pre-compute embeddings for all documents
        print(f"Encoding {len(documents)} documents...")
        self.embeddings = model.encode(documents, convert_to_tensor=True)
        print("Done!\n")

    def search(self, query, top_k=5):
        """
        Search for most similar documents to the query.
        """
        # Encode query
        query_embedding = self.model.encode(query, convert_to_tensor=True)

        # Compute similarities
        similarities = util.cos_sim(query_embedding, self.embeddings)[0]

        # Get top-k results
        top_results = similarities.argsort(descending=True)[:top_k]

        results = []
        for idx in top_results:
            results.append({
                'document': self.documents[idx],
                'similarity': similarities[idx].item()
            })

        return results

# Create a small document corpus
documents = [
    "Machine learning is a subset of artificial intelligence that enables computers to learn from data.",
    "Python is a popular programming language for data science and machine learning applications.",
    "Natural language processing helps computers understand and generate human language.",
    "Deep learning uses neural networks with multiple layers to solve complex problems.",
    "The weather today is sunny with a high temperature of 25 degrees Celsius.",
    "During the year, Barcelona has many sunny days.",
    "The stock market showed gains today as investors responded to positive economic news.",
    "Computer vision enables machines to interpret and understand visual information from the world.",
    "Data scientists use statistical methods to analyze and interpret complex datasets.",
    "Renewable energy sources like solar and wind power are becoming increasingly more attractive for investors."
]

# Create semantic search engine
search_engine = SemanticSearch(documents)

# Test queries
queries = [
    "How do computers learn?",
    "What programming language should I use for AI?",
    "Tell me about image recognition"
]

for query in queries:
    print(f"Query: '{query}'")
    print("=" * 80)

    results = search_engine.search(query, top_k=3)

    for i, result in enumerate(results, 1):
        print(f"{i}. [Score: {result['similarity']:.3f}] {result['document']}")
    print("\n")

## Visualizing Semantic Similarity

Let's visualize the similarity between multiple sentences using a heatmap:

In [ ]:
def plot_similarity_heatmap(sentences, model=sbert_model):
    """
    Create a heatmap showing pairwise similarities between sentences.
    """
    # Encode all sentences
    embeddings = model.encode(sentences, convert_to_tensor=True)

    # Compute pairwise similarities
    similarity_matrix = util.cos_sim(embeddings, embeddings).cpu().numpy()

    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 10))

    # Shorten labels for display
    labels = [s[:40] + "..." if len(s) > 43 else s for s in sentences]

    sns.heatmap(similarity_matrix,
                annot=True,
                fmt='.2f',
                cmap='Reds',
                xticklabels=labels,
                yticklabels=labels,
                cbar_kws={'label': 'Cosine Similarity'},
                vmin=0, vmax=1,
                linewidths=0.5,
                ax=ax)

    ax.set_title('Semantic Similarity Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Test sentences covering different topics
test_sentences = [
    "I love machine learning",
    "AI and ML are fascinating",
    "The weather is nice today",
    "It's sunny outside",
    "Python is a programming language",
    "I enjoy coding in Python",
]

plot_similarity_heatmap(test_sentences)

## Clustering Documents by Semantic Similarity

We can use semantic embeddings to cluster similar documents together:

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

def cluster_and_visualize(documents, n_clusters=3, model=sbert_model):
    """
    Cluster documents and visualize in 2D using PCA.
    """
    # Encode documents
    embeddings = model.encode(documents)

    # Reduce to 2D for visualization
    pca = PCA(n_components=2, random_state=42)
    embeddings_2d = pca.fit_transform(embeddings)

    # Cluster: two alternatives, from original embeddings or from 2D embeddings
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(embeddings_2d)

    # Plot
    fig, ax = plt.subplots(figsize=(14, 10))

    colors = plt.cm.Set3(np.linspace(0, 1, n_clusters))

    for i in range(n_clusters):
        cluster_points = embeddings_2d[clusters == i]
        ax.scatter(cluster_points[:, 0], cluster_points[:, 1],
                  c=[colors[i]], label=f'Cluster {i}', s=200, alpha=0.6, edgecolors='black')

    # Add labels
    for i, doc in enumerate(documents):
        label = doc[:40] + "..." if len(doc) > 40 else doc
        ax.annotate(label, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                   fontsize=8, alpha=0.8,
                   xytext=(5, 5), textcoords='offset points')

    ax.set_xlabel('First Principal Component', fontsize=12)
    ax.set_ylabel('Second Principal Component', fontsize=12)
    ax.set_title('Document Clustering Based on Semantic Similarity', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Print cluster assignments
    print("\nCluster Assignments:")
    print("=" * 80)
    for i in range(n_clusters):
        print(f"\nCluster {i}:")
        cluster_docs = [doc for doc, cluster in zip(documents, clusters) if cluster == i]
        for doc in cluster_docs:
            print(f"  - {doc}")

# Use our previous document corpus
cluster_and_visualize(documents, n_clusters=3)

# Part 3: NLP System Evaluation Metrics

Evaluation metrics help us assess the quality of NLP system outputs. Different tasks require different metrics. We'll cover:
- **BLEU** (Machine Translation)
- **ROUGE** (Summarization)
- **Task-specific metrics** (NER, QA, etc.)

## BLEU (Bilingual Evaluation Understudy)

**BLEU** is the standard metric for machine translation evaluation.

### Key Concepts:
- Measures **n-gram precision** (how many n-grams in the hypothesis appear in references)
- Uses **modified precision** to prevent gaming the metric
- Includes **brevity penalty** to discourage short translations
- Ranges from 0 to 1 (or 0 to 100)

### Formula:
$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where:
- $p_n$ = modified n-gram precision
- $w_n$ = uniform weights (typically $1/N$)
- $BP$ = brevity penalty

### Typical BLEU Scores:
- **< 10**: Essentially nonsense
- **10-20**: Hard to get the gist
- **20-30**: Clear idea of what the sentence is about
- **30-40**: Good translation
- **40-50**: Very good translation
- **50-60**: Extremely good, human-level
- **> 60**: Often better than human (possible overfitting)

In [ ]:
def calculate_bleu_detailed(reference, hypothesis):
    """
    Calculate BLEU score with detailed breakdown.
    """
    # Using sacrebleu for standardized BLEU
    bleu = sacrebleu.sentence_bleu(hypothesis, [reference])

    print(f"Reference:  {reference}")
    print(f"Hypothesis: {hypothesis}")
    print("=" * 70)
    print(f"BLEU Score: {bleu.score:.2f}")
    print(f"Precision breakdown:")
    for i, p in enumerate(bleu.precisions, 1):
        print(f"  {i}-gram precision: {p:.2f}")
    print(f"Brevity Penalty: {bleu.bp:.3f}")
    print(f"Length Ratio: {bleu.sys_len}/{bleu.ref_len} = {bleu.sys_len/bleu.ref_len:.3f}")
    print()

    return bleu.score

# Example 1: Good translation
print("Example 1: Good Translation")
print("=" * 70)
calculate_bleu_detailed(
    "The cat is on the mat",
    "The cat is on the mat"
)

# Example 2: Similar but not exact
print("Example 2: Similar Translation")
print("=" * 70)
calculate_bleu_detailed(
    "The cat is on the mat",
    "A cat sits on the mat"
)

# Example 3: Too short (brevity penalty)
print("Example 3: Too Short (Brevity Penalty Applied)")
print("=" * 70)
calculate_bleu_detailed(
    "The cat is sitting on the mat",
    "The cat"
)

# Example 4: Correct meaning, different words
print("Example 4: Same Meaning, Different Words")
print("=" * 70)
calculate_bleu_detailed(
    "The cat is on the mat",
    "A feline rests on the rug"
)

## Understanding N-gram Precision

In [ ]:
def analyze_ngram_overlap(reference, hypothesis, n=2):
    """
    Analyze and visualize n-gram overlap between reference and hypothesis.
    """
    from nltk import ngrams

    # Tokenize
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()

    # Generate n-grams
    ref_ngrams = list(ngrams(ref_tokens, n))
    hyp_ngrams = list(ngrams(hyp_tokens, n))

    # Find overlaps
    ref_ngram_set = set(ref_ngrams)
    hyp_ngram_set = set(hyp_ngrams)
    overlap = ref_ngram_set & hyp_ngram_set

    # Calculate precision
    precision = len(overlap) / len(hyp_ngrams) if hyp_ngrams else 0

    print(f"{n}-gram Analysis")
    print("=" * 70)
    print(f"Reference {n}-grams: {ref_ngrams}")
    print(f"Hypothesis {n}-grams: {hyp_ngrams}")
    print(f"Overlap: {list(overlap)}")
    print(f"Precision: {len(overlap)}/{len(hyp_ngrams)} = {precision:.3f}")
    print()

# Example
ref = "the cat sat on the mat"
hyp = "the cat is on the mat"

print(f"Reference:  {ref}")
print(f"Hypothesis: {hyp}")
print("\n")

for n in [1, 2, 3]:
    analyze_ngram_overlap(ref, hyp, n)

## BLEU with Multiple References

BLEU typically uses multiple reference translations to account for the fact that there are many valid ways to translate the same sentence.

In [ ]:
def compare_bleu_single_vs_multiple(references, hypothesis):
    """
    Compare BLEU scores with single vs multiple references.
    """
    print("Hypothesis:")
    print(f"  {hypothesis}")
    print("\nReferences:")
    for i, ref in enumerate(references, 1):
        print(f"  {i}. {ref}")
    print("\n" + "=" * 70)

    # BLEU with each single reference
    print("\nBLEU with single references:")
    for i, ref in enumerate(references, 1):
        score = sacrebleu.sentence_bleu(hypothesis, [ref]).score
        print(f"  Reference {i}: {score:.2f}")

    # BLEU with all references
    multi_score = sacrebleu.sentence_bleu(hypothesis, references).score
    print(f"\nBLEU with all references: {multi_score:.2f}")
    print()

# Example: Multiple valid translations
references = [
    "The cat is on the mat",
    "The cat sits on the mat",
    "There is a cat on the mat",
]
hypothesis = "A cat is sitting on the mat"

compare_bleu_single_vs_multiple(references, hypothesis)

## ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

**ROUGE** is the standard metric for summarization evaluation.

### Key Differences from BLEU:
- **Recall-based** rather than precision-based
- Better for summarization (want to capture key content)

### Common ROUGE Variants:
- **ROUGE-N**: N-gram overlap (ROUGE-1, ROUGE-2, etc.)
- **ROUGE-L**: Longest Common Subsequence
- **ROUGE-W**: Weighted Longest Common Subsequence

### Metrics:
- **Precision**: What fraction of generated n-grams appear in reference?
- **Recall**: What fraction of reference n-grams appear in generated text?
- **F1-score**: Harmonic mean of precision and recall

In [ ]:
def calculate_rouge_detailed(reference, hypothesis):
    """
    Calculate ROUGE scores with detailed breakdown.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)

    print(f"Reference:  {reference}")
    print(f"Hypothesis: {hypothesis}")
    print("=" * 70)

    for metric, score in scores.items():
        print(f"\n{metric.upper()}:")
        print(f"  Precision: {score.precision:.3f}")
        print(f"  Recall:    {score.recall:.3f}")
        print(f"  F1-Score:  {score.fmeasure:.3f}")
    print()

# Example 1: Good summary
print("Example 1: Complete Summary")
print("=" * 70)
reference = "The quick brown fox jumps over the lazy dog in the garden"
hypothesis = "The fox jumps over the dog"
calculate_rouge_detailed(reference, hypothesis)

# Example 2: Summary with additional info
print("Example 2: Summary with Extra Information")
print("=" * 70)
reference = "The cat sat on the mat"
hypothesis = "The fluffy cat sat on the red mat and purred loudly"
calculate_rouge_detailed(reference, hypothesis)

## Comparing BLEU and ROUGE

In [ ]:
def compare_bleu_rouge(reference, hypotheses):
    """
    Compare BLEU and ROUGE scores for different hypotheses.
    """
    print(f"Reference: {reference}")
    print("\n" + "=" * 90)
    print(f"{'Hypothesis':<60} {'BLEU':<10} {'ROUGE-1':<10} {'ROUGE-L':<10}")
    print("=" * 90)

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    for hyp in hypotheses:
        # BLEU
        bleu = sacrebleu.sentence_bleu(hyp, [reference]).score

        # ROUGE
        rouge_scores = scorer.score(reference, hyp)
        rouge1_f1 = rouge_scores['rouge1'].fmeasure
        rougeL_f1 = rouge_scores['rougeL'].fmeasure

        # Truncate hypothesis for display
        hyp_display = hyp[:55] + "..." if len(hyp) > 55 else hyp

        print(f"{hyp_display:<60} {bleu:>6.2f}     {rouge1_f1:>6.3f}     {rougeL_f1:>6.3f}")

# Test case: Different types of outputs
reference = "Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed"

hypotheses = [
    # Perfect match
    "Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed",
    # Good paraphrase
    "ML is part of AI and allows computers to learn from data",
    # Missing key information
    "Machine learning is part of artificial intelligence",
    # Too verbose
    "Machine learning, which is a subset of artificial intelligence, is a field that enables computers to learn from data without being explicitly programmed, and it's very important",
    # Wrong information
    "Machine learning is a type of database that stores computer programs",
]

compare_bleu_rouge(reference, hypotheses)

## Task-Specific Metrics

Different NLP tasks require different evaluation approaches. Let's implement metrics for common tasks.

### Named Entity Recognition (NER) Metrics

In [ ]:
def evaluate_ner(true_entities, predicted_entities):
    """
    Evaluate NER using precision, recall, and F1.

    Args:
        true_entities: Set of true entities (as tuples: (start, end, label))
        predicted_entities: Set of predicted entities
    """
    true_set = set(true_entities)
    pred_set = set(predicted_entities)

    # Calculate metrics
    tp = len(true_set & pred_set)  # True positives
    fp = len(pred_set - true_set)  # False positives
    fn = len(true_set - pred_set)  # False negatives

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print("NER Evaluation")
    print("=" * 70)
    print(f"True Positives:  {tp}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print()
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-Score:  {f1:.3f}")
    print()

    # Show errors
    if fp > 0:
        print("False Positives (incorrectly predicted):")
        for entity in (pred_set - true_set):
            print(f"  {entity}")
        print()

    if fn > 0:
        print("False Negatives (missed entities):")
        for entity in (true_set - pred_set):
            print(f"  {entity}")
        print()

    return precision, recall, f1

# Example: NER evaluation
# Format: (start_index, end_index, label, text)
true_entities = [
    (0, 12, 'PERSON', 'Barack Obama'),
    (31, 44, 'LOC', 'United States'),
    (50, 54, 'DATE', '2008'),
]

predicted_entities = [
    (0, 12, 'PERSON', 'Barack Obama'),  # Correct
    (31, 44, 'LOC', 'United States'),   # Correct
    (50, 54, 'CARDINAL', '2008'),       # Wrong label
    (20, 29, 'PERSON', 'President'),    # False positive
]

evaluate_ner(true_entities, predicted_entities)

### Question Answering (QA) Metrics

In [ ]:
def evaluate_qa(reference_answer, predicted_answer):
    """
    Evaluate QA using exact match and F1 score (token-level).
    """
    # Exact Match
    exact_match = 1.0 if reference_answer.lower().strip() == predicted_answer.lower().strip() else 0.0

    # Token-level F1
    ref_tokens = reference_answer.lower().split()
    pred_tokens = predicted_answer.lower().split()

    common = Counter(ref_tokens) & Counter(pred_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        f1 = 0.0
    else:
        precision = num_same / len(pred_tokens)
        recall = num_same / len(ref_tokens)
        f1 = 2 * precision * recall / (precision + recall)

    print(f"Reference: {reference_answer}")
    print(f"Predicted: {predicted_answer}")
    print("=" * 70)
    print(f"Exact Match: {exact_match}")
    print(f"Token F1:    {f1:.3f}")
    print()

    return exact_match, f1

# Examples
print("Example 1: Exact Match")
print("=" * 70)
evaluate_qa("Paris", "Paris")

print("Example 2: Partial Match")
print("=" * 70)
evaluate_qa("the capital of France", "capital of France")

print("Example 3: Paraphrase")
print("=" * 70)
evaluate_qa("Albert Einstein", "Einstein")

## Limitations of Automatic Metrics

Let's demonstrate some key limitations of automatic metrics:

In [ ]:
def demonstrate_metric_limitations():
    """
    Show cases where automatic metrics fail to capture quality.
    """
    print("LIMITATIONS OF AUTOMATIC METRICS")
    print("=" * 90)

    # Case 1: Semantically incorrect but high BLEU
    print("\nCase 1: High lexical overlap, wrong meaning")
    print("-" * 90)
    ref = "The stock price increased significantly"
    hyp = "The stock price decreased significantly"  # Opposite meaning!

    bleu = sacrebleu.sentence_bleu(hyp, [ref]).score
    sem_sim, _, _ = compute_semantic_similarity(ref, hyp)

    print(f"Reference:  {ref}")
    print(f"Hypothesis: {hyp}")
    print(f"BLEU Score: {bleu:.2f} (HIGH - but meaning is opposite!)")
    print(f"Semantic Similarity: {sem_sim:.3f} (correctly shows dissimilarity)")

    # Case 2: Correct meaning but low BLEU
    print("\n" + "=" * 90)
    print("Case 2: Low lexical overlap, correct meaning")
    print("-" * 90)
    ref = "The movie was excellent"
    hyp = "The film was outstanding"  # Same meaning, different words

    bleu = sacrebleu.sentence_bleu(hyp, [ref]).score
    sem_sim, _, _ = compute_semantic_similarity(ref, hyp)

    print(f"Reference:  {ref}")
    print(f"Hypothesis: {hyp}")
    print(f"BLEU Score: {bleu:.2f} (LOW - but meaning is correct!)")
    print(f"Semantic Similarity: {sem_sim:.3f} (correctly shows similarity)")

    # Case 3: Fluent but nonsensical
    print("\n" + "=" * 90)
    print("Case 3: Fluent but nonsensical (word salad)")
    print("-" * 90)
    ref = "The cat sat on the mat"
    hyp = "The mat sat on the cat"  # Grammatical but wrong

    bleu = sacrebleu.sentence_bleu(hyp, [ref]).score

    print(f"Reference:  {ref}")
    print(f"Hypothesis: {hyp}")
    print(f"BLEU Score: {bleu:.2f} (HIGH - but semantically wrong!)")

demonstrate_metric_limitations()

 Automatic metrics are useful but imperfect, they should be complemented with human evaluation when possible.

## Modern Evaluation: BERTScore

**BERTScore** uses contextual embeddings (BERT) to evaluate similarity, addressing some limitations of n-gram metrics.

### Advantages:
- Captures semantic similarity
- Handles paraphrases better
- More robust to word order changes

In [ ]:
from transformers import BertTokenizer, BertModel
from bert_score import BERTScorer

# BERTScore calculation
bert_scorer = BERTScorer(model_type='bert-base-uncased')

In [ ]:
def compare_traditional_vs_semantic_metrics(reference, hypotheses):
    """
    Compare traditional metrics (BLEU, ROUGE) with semantic similarity.
    """
    print(f"Reference: {reference}")
    print("\n" + "=" * 100)
    print(f"{'Hypothesis':<50} {'BLEU':<8} {'ROUGE-L':<10} {'BERTScore(P)':<10}")
    print("=" * 100)

    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    for hyp in hypotheses:
        # Traditional metrics
        bleu = sacrebleu.sentence_bleu(hyp, [reference]).score
        rouge = scorer.score(reference, hyp)['rougeL'].fmeasure

        # Semantic similarity
        P, R, F1 = bert_scorer.score([hyp], [reference])

        # Truncate for display
        hyp_display = hyp[:47] + "..." if len(hyp) > 50 else hyp

        print(f"{hyp_display:<50} {bleu:>6.2f}   {rouge:>6.3f}     {P.mean():>6.3f}")

# Test cases that highlight differences
reference = "The experiment was successful and the results were positive"

hypotheses = [
    "The experiment was successful and the results were positive",  # Exact match
    "The test succeeded with good outcomes",  # Paraphrase
    "The experiment failed and the results were negative",  # Opposite meaning
    "Successful experiment positive results",  # Keyword match, no grammar
    "The weather was nice and sunny today",  # Completely different
]

compare_traditional_vs_semantic_metrics(reference, hypotheses)